### Models

Purpose : Isolate modelling in one file 

In [1]:


import os, sys
sys.path.insert(0, "/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")
sys.modules.pop("srcA", None)
os.chdir("/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")
sys.modules.pop("srcA", None)
import pandas as pd
import numpy as np
from math import sqrt, pi, exp
from arch import arch_model
import yfinance as yf 
from scipy.optimize import minimize
import warnings
from arch.__future__ import reindexing
from arch.utility.exceptions import ConvergenceWarning
# from srcA.msGarch import msGARCH
# from srcA.metrics import metrics
# from srcA.msGarch import msGARCHIV
from tqdm import tqdm

from xgboost import XGBRegressor
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="y is poorly scaled")
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

import sys, os
sys.path.insert(0, "/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/MFIV_calibration")
from srcA.models_MFIV import estimate_naive10_mfiv, estimate_garch_mfiv

In [2]:
sys.modules.pop("srcA", None)

<module 'srcA' (namespace) from ['/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/MFIV_calibration/srcA']>

In [3]:
import pandas as pd
with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/x_by_year/x_by_year.pkl", "rb") as f:
    X_by_year = pd.read_pickle(f)

In [26]:
X_by_year[2020].columns

Index(['rt', 'bsIV_191', 'MFIV_191', 'bsIV_382', 'MFIV_382', 'bsIV_573',
       'MFIV_573', 'bsIV_764', 'MFIV_764', 'mfiv_avg', 'rv'],
      dtype='str')

In [5]:
YEARS = [2019, 2020, 2021]
for y in YEARS:
    X_by_year[y] = X_by_year[y].copy()
    X_by_year[y]["rv"] = X_by_year[y]["rt"] ** 2

### quarter split

In [6]:
# Just prepping for the short parameter tuning 

Xq_by_year = {}
YEARS = [2019, 2020, 2021]
for y in YEARS:
    Xq_by_year[y] = {}
    Xq_by_year[y]["q1"] = X_by_year[y].loc[f"{y}-01-01" : f"{y}-03-31 23:59:59"]
    Xq_by_year[y]["q2"] = X_by_year[y].loc[f"{y}-04-01" : f"{y}-06-30 23:59:59"]
    Xq_by_year[y]["q3"] = X_by_year[y].loc[f"{y}-07-01" : f"{y}-09-30 23:59:59"]
    Xq_by_year[y]["q4"] = X_by_year[y].loc[f"{y}-10-01" : f"{y}-12-31 23:59:59"]


### Outputs By Year

In [7]:
import pandas as pd
YEARS = [2019, 2020, 2021]
QUARTER = ["q1", "q2", "q3", "q4"]
outputs_by_year = {
    y : {
        q : pd.DataFrame(
            {
                "rv" : Xq_by_year[y][q]["rv"],
                "rt" : Xq_by_year[y][q]["rt"],
                "date_time" : Xq_by_year[y][q].index,
                
            }
        ) for q in QUARTER
    } for y in YEARS
}

In [27]:
YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]
# HOURS = [7, 46, 191]
HOURS = [191, 382, 573, 764]
for y in YEARS:
    for q in QUARTERS:
        cols = [f"MFIV_{H}" for H in HOURS]
        outputs_by_year[y][q] = pd.concat([outputs_by_year[y][q], Xq_by_year[y][q][cols]], axis = 1)

###  garch

In [8]:
from srcA.models import estimate_garch 

In [9]:
# # Prepare the dictionaries. Designed to be overwritten
# YEARS = [2022, 2023, 2024, 2025]
# QUARTERS = ["q1", "q2", "q3", "q4"]

YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]

idx_by_year = {y : {} for y in YEARS}
f_rollB_by_year = {y: {} for y in YEARS}
rolling_forecasts =  {y: {} for y in YEARS}
rolling_residuals = {y: {} for y in YEARS}
y_pred_by_year = {y: {} for y in YEARS}
y_true_by_year = {y: {} for y in YEARS}


In [10]:

for y in YEARS:
    for q in QUARTERS:

        Xq_df = Xq_by_year[y][q]

        f_rollB, y_true, idx = estimate_garch(Xq_df, lookback = 10)

        dt = Xq_df.index
        x_plotB = dt[idx]

        assert len(x_plotB) == len(f_rollB) == len(y_true), (y, q)

        idx_by_year[y][q] = idx
        f_rollB_by_year[y][q] = f_rollB
        rolling_forecasts[y][q] = f_rollB
        rolling_residuals[y][q] = y_true - f_rollB
        y_pred_by_year[y][q] = f_rollB
        y_true_by_year[y][q] = y_true
        
        outputs_by_year[y][q]["garch"] = pd.Series(f_rollB, Xq_by_year[y][q].index[idx])

In [11]:
outputs_by_year[y][q]

,rv,rt,date_time,garch
date_time,,,,
2021-10-01 00:00:00+00:00,2.974385e-06,0.001725,2021-10-01 00:00:00+00:00,NaN
2021-10-01 01:00:00+00:00,1.053757e-05,-0.003246,2021-10-01 01:00:00+00:00,NaN
2021-10-01 02:00:00+00:00,2.905000e-06,0.001704,2021-10-01 02:00:00+00:00,NaN
2021-10-01 03:00:00+00:00,5.339294e-05,-0.007307,2021-10-01 03:00:00+00:00,NaN
2021-10-01 04:00:00+00:00,3.645259e-05,0.006038,2021-10-01 04:00:00+00:00,NaN
...,...,...,...,...
2021-12-31 19:00:00+00:00,6.898782e-05,-0.008306,2021-12-31 19:00:00+00:00,0.000101
2021-12-31 20:00:00+00:00,3.150191e-04,-0.017749,2021-12-31 20:00:00+00:00,0.000039
2021-12-31 21:00:00+00:00,3.557233e-06,0.001886,2021-12-31 21:00:00+00:00,0.000038


In [12]:
os.makedirs("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/outputs_by_year", exist_ok = True)

In [ ]:
# import pickle
# with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/outputs_by_year/outputs_by_year_01.pkl", "wb") as f:
#     pickle.dump(outputs_by_year, f)


load

In [ ]:
import pickle 
with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/outputs_by_year/outputs_by_year_01.pkl", "rb") as f:
    outputs_by_year  = pickle.load(f)

In [14]:

outputs_by_year[2021]["q1"]

,rv,rt,date_time,garch
date_time,,,,
2021-01-01 00:00:00+00:00,1.161073e-05,-0.003407,2021-01-01 00:00:00+00:00,NaN
2021-01-01 01:00:00+00:00,1.454392e-06,0.001206,2021-01-01 01:00:00+00:00,NaN
2021-01-01 02:00:00+00:00,2.782889e-04,0.016682,2021-01-01 02:00:00+00:00,NaN
2021-01-01 03:00:00+00:00,6.549349e-05,-0.008093,2021-01-01 03:00:00+00:00,NaN
2021-01-01 04:00:00+00:00,1.173290e-05,0.003425,2021-01-01 04:00:00+00:00,NaN
...,...,...,...,...
2021-03-31 19:00:00+00:00,4.932710e-06,0.002221,2021-03-31 19:00:00+00:00,0.000034
2021-03-31 20:00:00+00:00,2.861346e-04,-0.016916,2021-03-31 20:00:00+00:00,0.000030
2021-03-31 21:00:00+00:00,1.207885e-04,0.010990,2021-03-31 21:00:00+00:00,0.000101


### garch MFIV_191


In [15]:
# # Prepare the dictionaries. Designed to be overwritten
# YEARS = [2022, 2023, 2024, 2025]
# QUARTERS = ["q1", "q2", "q3", "q4"]

YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]

idx_by_year = {y : {} for y in YEARS}
f_rollB_by_year = {y: {} for y in YEARS}
rolling_forecasts =  {y: {} for y in YEARS}
rolling_residuals = {y: {} for y in YEARS}
y_pred_by_year = {y: {} for y in YEARS}
y_true_by_year = {y: {} for y in YEARS}


In [16]:
from srcA.models_MFIV import estimate_garch_mfiv

In [17]:
YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]

for y in YEARS:
    for q in QUARTERS:
        Xq_df = Xq_by_year[y][q]

        f_rollB, y_true, idx = estimate_garch_mfiv(
            Xq_df,
            exog_col="MFIV_191",
            lookback=10,
            correction_lookback=10
        )

        idx_by_year[y][q] = idx
        f_rollB_by_year[y][q] = f_rollB
        rolling_forecasts[y][q] = f_rollB
        rolling_residuals[y][q] = y_true - f_rollB
        y_pred_by_year[y][q] = f_rollB
        y_true_by_year[y][q] = y_true

        outputs_by_year[y][q]["garch_mfiv"] = pd.Series(
            f_rollB,
            index=Xq_df.index[idx]
        )

In [18]:
import pickle
with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/outputs_by_year_02.pkl", "wb") as f:
    pickle.dump(outputs_by_year, f)


load

In [12]:
import pickle 
with open("/Users/shah/CODE_BOOK_4/CodeCleanup_II/script_data/IS/outputs_by_year/outputs_by_year_02.pkl", "rb") as f:
    outputs_by_year = pickle.load(f)

### garch MFIV avg

In [19]:
# # Prepare the dictionaries. Designed to be overwritten
# YEARS = [2022, 2023, 2024, 2025]
# QUARTERS = ["q1", "q2", "q3", "q4"]

YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]

idx_by_year = {y : {} for y in YEARS}
f_rollB_by_year = {y: {} for y in YEARS}
rolling_forecasts =  {y: {} for y in YEARS}
rolling_residuals = {y: {} for y in YEARS}
y_pred_by_year = {y: {} for y in YEARS}
y_true_by_year = {y: {} for y in YEARS}


In [20]:
from srcA.models_MFIV import estimate_garch_mfiv

In [21]:
YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]

for y in YEARS:
    for q in QUARTERS:
        Xq_df = Xq_by_year[y][q]

        f_rollB, y_true, idx = estimate_garch_mfiv(
            Xq_df,
            exog_col="mfiv_avg",
            lookback=10,
            correction_lookback=10
        )

        idx_by_year[y][q] = idx
        f_rollB_by_year[y][q] = f_rollB
        rolling_forecasts[y][q] = f_rollB
        rolling_residuals[y][q] = y_true - f_rollB
        y_pred_by_year[y][q] = f_rollB
        y_true_by_year[y][q] = y_true

        outputs_by_year[y][q]["garch_mfiv_avg"] = pd.Series(
            f_rollB,
            index=Xq_df.index[idx]
        )

In [23]:
# NOW CONTAINS THREE MODELS, garch, garch_mfiv(191), garch_mfiv_avg
import pickle
with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/outputs_by_year/outputs_by_year_03.pkl", "wb") as f:
    pickle.dump(outputs_by_year, f)


In [ ]:
import pickle 
with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/outputs_by_year/outputs_by_year_03", "rb") as f:
    outputs_by_year = pickle.load(f)

In [25]:
outputs_by_year[2020]["q1"][20:200]

,rv,rt,date_time,garch,garch_mfiv,garch_mfiv_avg
date_time,,,,,,
2020-01-01 20:00:00+00:00,1.732862e-06,-0.001316,2020-01-01 20:00:00+00:00,0.000003,NaN,NaN
2020-01-01 21:00:00+00:00,9.411596e-07,0.000970,2020-01-01 21:00:00+00:00,0.000003,NaN,NaN
2020-01-01 22:00:00+00:00,3.888139e-07,-0.000624,2020-01-01 22:00:00+00:00,0.000003,NaN,NaN
2020-01-01 23:00:00+00:00,3.014151e-05,-0.005490,2020-01-01 23:00:00+00:00,0.000003,NaN,NaN
2020-01-02 00:00:00+00:00,0.000000e+00,0.000000,2020-01-02 00:00:00+00:00,0.000004,NaN,NaN
...,...,...,...,...,...,...
2020-01-09 03:00:00+00:00,1.132088e-06,-0.001064,2020-01-09 03:00:00+00:00,0.000092,0.000093,0.000109
2020-01-09 04:00:00+00:00,1.023231e-05,-0.003199,2020-01-09 04:00:00+00:00,0.000037,0.000002,0.000002
2020-01-09 05:00:00+00:00,2.524555e-07,0.000502,2020-01-09 05:00:00+00:00,0.000027,0.000013,0.000011


In [28]:
outputs_by_year[2021]["q4"][["MFIV_191", "MFIV_382", "MFIV_573", "MFIV_764"]].notna().sum()

MFIV_191    2208
MFIV_382    2208
MFIV_573    2208
MFIV_764    2208
dtype: int64

### GARCH regime switching

In [29]:
sys.modules.pop("srcA", None)

<module 'srcA' (namespace) from ['/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/MFIV_calibration/srcA']>

In [30]:
# # Prepare the dictionaries. Designed to be overwritten

YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]

idx_by_year = {y : {} for y in YEARS}
f_rollB_by_year = {y: {} for y in YEARS}
rolling_forecasts =  {y: {} for y in YEARS}
rolling_residuals = {y: {} for y in YEARS}
y_pred_by_year = {y: {} for y in YEARS}
y_true_by_year = {y: {} for y in YEARS}


In [31]:
# EXPERIMENTAL 04
def regime_switching_mfiv(
    Xq_df,
    MFIV_COLS=["MFIV_191", "MFIV_382", "MFIV_573", "MFIV_764"],
    return_col="rt",
    target_col="rv",
    refit_every=1,
    min_improvement=0.02
):
    rv = Xq_df[target_col].astype(float)
    mfiv = Xq_df[list(MFIV_COLS)].astype(float)

    horizons = {
        col: int(col.split("_")[1])
        for col in MFIV_COLS
    }

    # For an MFIV forecast made at time t:
    #
    # matched_future_rv[col][t]
    #     = rv[t+1] + ... + rv[t+H]
    #
    # This is used only for historical forecast evaluation.
    matched_future_rv = {}

    for col, H in horizons.items():
        matched_future_rv[col] = (
            rv.shift(-1)
              .rolling(window=H, min_periods=H)
              .sum()
              .shift(-(H - 1))
        )

    f_rollB = []
    y_true = []
    idx = []
    regime_state = []
    selected_horizon = []

    selected_col = None

    # The longest horizon needs:
    # H hours for each completed outcome
    # plus H completed forecast-outcome observations.
    start = 2 * max(horizons.values())

    for i in range(start, len(Xq_df)):

        if selected_col is None or (i - start) % refit_every == 0:
            losses = {}

            for col, H in horizons.items():

                # At time i, rv[i] is not assumed available.
                # Therefore the latest completed H-hour outcome must
                # finish at or before i - 1.
                #
                # A forecast originating at u finishes at u + H.
                # Therefore u <= i - H - 1.
                history_end = i - H

                forecast_history = mfiv[col].iloc[:history_end]
                realised_history = matched_future_rv[col].iloc[:history_end]

                valid = (
                    forecast_history.notna()
                    & realised_history.notna()
                    & (forecast_history > 0)
                    & (realised_history > 0)
                )

                forecast_history = forecast_history.loc[valid].iloc[-H:]
                realised_history = realised_history.loc[valid].iloc[-H:]

                # Each horizon is evaluated using H completed
                # forecast-outcome pairs.
                if len(forecast_history) < H:
                    losses[col] = np.nan
                    continue

                ratio = realised_history / forecast_history

                # QLIKE divergence.
                qlike = ratio - np.log(ratio) - 1

                losses[col] = float(qlike.mean())

            losses = pd.Series(losses, dtype=float).dropna()

            if not losses.empty:
                best_col = losses.idxmin()
                best_loss = losses.loc[best_col]

                if selected_col is None:
                    selected_col = best_col

                elif selected_col not in losses.index:
                    selected_col = best_col

                else:
                    current_loss = losses.loc[selected_col]

                    # Avoid switching because of a negligible
                    # difference in historical loss.
                    if best_loss < current_loss * (1 - min_improvement):
                        selected_col = best_col

        if selected_col is None:
            continue

        forecast = mfiv[selected_col].iloc[i]

        if pd.isna(forecast) or forecast <= 0:
            continue

        matched_outcome = matched_future_rv[selected_col].iloc[i]

        f_rollB.append(float(forecast))

        # Future realised variance is used only as the evaluation target.
        # It is not used when selecting the regime at time i.
        y_true.append(
            float(matched_outcome)
            if pd.notna(matched_outcome)
            else np.nan
        )

        idx.append(i)
        regime_state.append(MFIV_COLS.index(selected_col))
        selected_horizon.append(selected_col)

    # return (
    #     np.array(f_rollB),
    #     np.array(y_true),
    #     np.array(idx),
    #     np.array(regime_state),
    #     np.array(selected_horizon)
    # )
        
    return (
        np.asarray(f_rollB, dtype=float),
        np.asarray(y_true, dtype=float),
        np.asarray(idx, dtype=int),
        np.asarray(regime_state, dtype=int),
        np.asarray(selected_horizon, dtype=object)
    )

This model is a dynamic MFIV horizon selector, not a GARCH model and not a regression model.

It:
- compares four MFIV forecasts: MFIV_191, MFIV_382, MFIV_573, and MFIV_764;
- treats each MFIV_H[t] as a forecast of cumulative realised variance over t+1 through t+H;
- evaluates each forecast only after that future realised variance is fully known;
- uses QLIKE loss: realised / forecast - log(realised / forecast) - 1;
- evaluates MFIV_H using its latest H valid, positive, completed forecast-outcome pairs;
- begins producing results only after 2 × max(H) rows;
- reassesses the selected horizon every row because refit_every=1;
- selects the MFIV horizon with the lowest historical average QLIKE loss;
- keeps the current horizon unless the new horizon improves loss by more than 2%;
- outputs the selected MFIV value as f_rollB;
- outputs its matched future cumulative realised variance as y_true, which may be NaN near the end;
- outputs the integer row positions as idx;
- outputs the selected horizon’s list index as regime_state;
- outputs the actual column name, such as MFIV_382, as selected_horizon.


This function’s job remains:
- select the best MFIV_H,
- output the selected MFIV forecast,
- output regime_state,
- output selected_horizon.

In [32]:
YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]

for y in YEARS:
    for q in QUARTERS:
        Xq_df = Xq_by_year[y][q]

        f_rollB, y_true, idx, regime_state, selected_horizon = regime_switching_mfiv(
            Xq_df
        )

        idx_by_year[y][q] = idx
        f_rollB_by_year[y][q] = f_rollB
        rolling_forecasts[y][q] = f_rollB
        rolling_residuals[y][q] = y_true - f_rollB
        y_pred_by_year[y][q] = f_rollB
        y_true_by_year[y][q] = y_true

        outputs_by_year[y][q]["regime_switching_mfiv"] = pd.Series(
            f_rollB,
            index=Xq_df.index[idx]
        )

        outputs_by_year[y][q]["regime_state"] = pd.Series(
            regime_state,
            index=Xq_df.index[idx]
        )

        outputs_by_year[y][q]["selected_horizon"] = pd.Series(
            selected_horizon,
            index=Xq_df.index[idx]
        )

In [40]:
outputs_by_year[2021]["q4"]["selected_horizon"].nunique()

1

In [41]:
outputs_by_year[2020]["q4"].columns

Index(['rv', 'rt', 'date_time', 'garch', 'garch_mfiv', 'garch_mfiv_avg',
       'MFIV_191', 'MFIV_382', 'MFIV_573', 'MFIV_764', 'regime_switching_mfiv',
       'regime_state', 'selected_horizon'],
      dtype='str')

In [42]:
import pickle
with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/outputs_by_year/outputs_by_year_04.pkl", "wb") as f:
    pickle.dump(outputs_by_year, f)